In [19]:
import csv
import math
import re
import os
import time
import pandas as pd
from transformers import pipeline

SCRAPED_FILE = "webscraped.csv"     
ANALYSIS_FILE = "textanalysis.csv"  
MAX_HF_INPUT_LENGTH = 512           


df = pd.read_csv(SCRAPED_FILE)
available_columns = df.columns.tolist()

comment_column = "selftext" if "selftext" in available_columns else available_columns[-1]


def summarize_lsd(text, max_sentences=3):

    text = text.replace("\n", " ")
    sentences = re.split(r'(?<=[.!?]) +', text)

    if len(sentences) <= max_sentences:
        return text


    word_freq = {}
    for sentence in sentences:
        words = re.findall(r"\w+", sentence.lower())
        for w in words:
            word_freq[w] = word_freq.get(w, 0) + 1


    sentence_scores = [(sentence, sum(word_freq.get(w, 0) for w in re.findall(r"\w+", sentence.lower())))
                       for sentence in sentences]


    top_sentences = sorted(sentence_scores, key=lambda x: x[1], reverse=True)[:max_sentences]
    summary = " ".join([s[0] for s in sorted(top_sentences, key=lambda x: sentences.index(x[0]))])
    
    return summary

def get_importance_and_direction_lsd(text):

    word_count = len(re.findall(r"\w+", text))
    importance_score = 1 if word_count > 50 else -1
    direction = "Positive" if importance_score == 1 else "Negative"
    return importance_score, direction


summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_hf(text):

    words = text.split()
    if len(words) < 30:
        return text 

    truncated_text = " ".join(words[:MAX_HF_INPUT_LENGTH])
    try:
        summary = summarizer(truncated_text, max_length=150, min_length=50, do_sample=False)
        return summary[0]['summary_text']
    except Exception as e:
        return f"Summarization error: {str(e)}"

def get_importance_and_direction_hf(text):

    word_count = len(re.findall(r"\w+", text))
    importance_score = 1 if word_count % 2 == 0 else -1
    direction = "Positive" if importance_score == 1 else "Negative"
    return importance_score, direction



def run_text_analysis(scraped_file, analysis_file):

    results = []
    start_time = time.time()
    
    with open(scraped_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader, start=1):
            title = row["title"]
            comments_text = row.get(comment_column, "No comments available.")

            print(f"Processing post {i}/100: {title[:50]}...")

            
            lsd_summary = summarize_lsd(comments_text)
            lsd_importance, lsd_direction = get_importance_and_direction_lsd(comments_text)

            
            hf_summary = summarize_hf(comments_text)
            hf_importance, hf_direction = get_importance_and_direction_hf(comments_text)

            results.append({
                "title": title,
                "lsd_summary": lsd_summary,
                "lsd_importance_score": lsd_importance,
                "lsd_direction": lsd_direction,
                "hf_summary": hf_summary,
                "hf_importance_score": hf_importance,
                "hf_direction": hf_direction
            })

    
    fieldnames = [
        "title",
        "lsd_summary", "lsd_importance_score", "lsd_direction",
        "hf_summary", "hf_importance_score", "hf_direction"
    ]
    
    with open(analysis_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow(r)

    elapsed_time = time.time() - start_time  
    print(f"Text analysis complete in {elapsed_time:.2f} seconds! Results saved to {analysis_file}")


if __name__ == "__main__":
    print(f"Running text analysis on {SCRAPED_FILE}...")
    run_text_analysis(SCRAPED_FILE, ANALYSIS_FILE)
    print("Processing complete! Check textanalysis.csv for results.")


Device set to use cpu


Running text analysis on webscraped.csv...
Processing post 1/100: ‘Only Works as a State’: Trump Vows Not ‘To Bend’ ...
Processing post 2/100: Statement by the Prime Minister on unjustified U.S...
Processing post 3/100: Canada retaliating for Trump’s tariffs with 25 per...
Processing post 4/100: Jack Daniel’s maker says Canada pulling U.S. alcoh...
Processing post 5/100: “It’s done, it’s gone”: Ontario Premier Doug Ford ...
Processing post 6/100: Trump threatens Canadian cars with tariffs up to 1...
Processing post 7/100: Trump threatens new tariffs on Canada, including 2...
Processing post 8/100: U.S. tariffs will be imposed on Feb. 4...
Processing post 9/100: Hockey fans boo U.S. national anthem at Ottawa Sen...
Processing post 10/100: Furious Poilievre criticizes Trump tariffs for uni...
Processing post 11/100: Canada Won’t Scrap Tariffs Unless All US Levies Ar...
Processing post 12/100: Chrystia Freeland says Canada should target Elon M...
Processing post 13/100: Kentucky governor 